# Aula 02 - Notebook: Mapeamento de Variáveis de Processo em Proposições Lógicas

Neste notebook implementamos a camada de conversão de corrente $4 \dots 20	ext{ mA}$ para unidades de engenharia, funções de discretização com validação de limites de processo e a geração do vetor proposicional completo para a planta de fertilizantes.


In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

from typing import Dict, List, Tuple

class ConversorSinal4a20mA:
    @staticmethod
    def corrente_para_engenharia(i_mA: float, min_eng: float, max_eng: float) -> Tuple[float, str]:
        if i_mA < 3.6:
            return min_eng, "FALHA_LOOP_ROMPIDO (NAMUR < 3.6mA)"
        elif i_mA > 21.0:
            return max_eng, "FALHA_CURTO_CIRCUITO (NAMUR > 21.0mA)"
        
        i_clip = max(4.0, min(20.0, i_mA))
        fator = (i_clip - 4.0) / 16.0
        valor_eng = min_eng + fator * (max_eng - min_eng)
        return valor_eng, "OK"

class MapeadorProposicional:
    def __init__(self):
        self.P_CRIT = 180.0
        self.T_CRIT = 200.0
        self.GAS_CRIT = 25.0
        self.NIVEL_HIGH = 90.0
        self.NIVEL_LOW = 20.0
        self.FLOW_MIN = 600.0
        
    def extrair_proposicoes(self, telemetria: Dict[str, float]) -> Dict[str, bool]:
        return {
            'p1': telemetria.get('PT-101', 0.0) >= self.P_CRIT,
            't1': telemetria.get('TT-101', 0.0) >= self.T_CRIT,
            'g1': telemetria.get('AT-101', 0.0) >= self.GAS_CRIT,
            'l_high': telemetria.get('LT-101', 0.0) >= self.NIVEL_HIGH,
            'l_low': telemetria.get('LT-101', 0.0) <= self.NIVEL_LOW,
            'e1': bool(telemetria.get('ESD-100', 0)),
            'v1': bool(telemetria.get('XV-101', 0)),
            'v2': bool(telemetria.get('XV-102', 0)),
            'm1': bool(telemetria.get('AG-101', 0)),
            'a1': bool(telemetria.get('ALM-101', 0)),
            'f1': telemetria.get('FS-201', 0.0) >= self.FLOW_MIN,
            'c1': bool(telemetria.get('TS-201', 0)),
            'm2': bool(telemetria.get('M-201', 0)),
            'v3': bool(telemetria.get('XV-201', 0))
        }

mapeador = MapeadorProposicional()
amostra_campo = {
    'PT-101': 195.4,
    'TT-101': 160.0,
    'AT-101': 4.2,
    'LT-101': 68.0,
    'ESD-100': 0,
    'XV-101': 1,
    'XV-102': 1,
    'AG-101': 1,
    'ALM-101': 0,
    'FS-201': 1100.0,
    'TS-201': 1,
    'M-201': 1,
    'XV-201': 1
}

props = mapeador.extrair_proposicoes(amostra_campo)
tabela_p = [{"Proposição": k, "Valor Booleano": v, "Status": "CRÍTICO/ATIVO" if v else "NORMAL/INATIVO"} for k, v in props.items()]
print(formatar_tabela(tabela_p))

assert props['p1'] is True
assert props['t1'] is False
print("\n[OK] Módulo de Mapeamento de Proposições validado com sucesso!")


Proposição | Valor Booleano | Status        
-----------+----------------+---------------
p1         | True           | CRÍTICO/ATIVO 
t1         | False          | NORMAL/INATIVO
g1         | False          | NORMAL/INATIVO
l_high     | False          | NORMAL/INATIVO
l_low      | False          | NORMAL/INATIVO
e1         | False          | NORMAL/INATIVO
v1         | True           | CRÍTICO/ATIVO 
v2         | True           | CRÍTICO/ATIVO 
m1         | True           | CRÍTICO/ATIVO 
a1         | False          | NORMAL/INATIVO
f1         | True           | CRÍTICO/ATIVO 
c1         | True           | CRÍTICO/ATIVO 
m2         | True           | CRÍTICO/ATIVO 
v3         | True           | CRÍTICO/ATIVO 

[OK] Módulo de Mapeamento de Proposições validado com sucesso!
